In [1]:
import os

import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import (
    get_tracking_uri,
    log_metric,
    log_params,
    set_experiment,
    set_tracking_uri,
    start_run,
)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest as IsoF
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from skopt import BayesSearchCV
from skopt.space import Integer

import dataprocessing as dc


In [2]:
df = pd.read_csv("data/email_data.csv")
newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = np.where(df['Category'] == 'spam', 1, 0)


In [3]:
#splitting data into spam with 80-20 split and real messages
dfspam = df[df["Category"] == 1]
dftrue = df[df["Category"] == 2]


In [4]:
y = dfspam.loc[:, ["Category"]]
X = dfspam.drop(
    ["Category", "Message", "word_count"],
    axis = 1
)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size = 0.2   # Reserve 20% for "hold-out" data
)
y2 = dftrue.loc[:, ["Category"]]
X2 = dftrue.drop(
    ["Category", "Message", "word_count"],
    axis = 1
)
# Stack the DataFrames on top of each other
X_test = pd.concat([X_temp, X2], axis=0)
y_test = pd.concat([y_temp, y2], axis=0)




In [5]:
numeric_cols = X.select_dtypes(
    include = ["number"]
).columns
numeric_pipe = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="mean")),
        ("encoder",MinMaxScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_cols)
    ],
    remainder='drop'  # or 'passthrough' if you want to keep unlisted columns
)

In [6]:
model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("model", IsoF(random_state = 42))
    ]
)

In [7]:
n_features = X_train.shape[1]
search_space = {
    "model__n_estimators": Integer(100,500),
    "model__max_samples": ['auto', 64, 128, 256, 512],
    "model__max_features": [1, .5, int(n_features ** 0.5), int(np.log2(n_features))],
}

In [ ]:
#%%capture #uncomment to supress output
with start_run(run_name="IsolationForest_BayesianSearch_AllConfigs"):
    
    # Declare 5-Folds CV Grid Search within pipeline
    bayes_search_cv = BayesSearchCV(
    estimator=model, search_spaces=search_space,
    n_iter=64,               # Number of search iterations
    cv=5,                    # Cross-validation folds
    scoring='f1',      # Can be any valid metric    accuracy 
    random_state=42,
    verbose=0
)
    bayes_search_cv.fit(X_train, y_train)  # Train algorithm with CV

    # Log top-level best model including metrics & parameters
    log_params(bayes_search_cv.best_params_)
    log_metric("best_score_cv", bayes_search_cv.best_score_)
    mlflow.sklearn.log_model(bayes_search_cv.best_estimator_, name = "best_model", serialization_format="cloudpickle")

    # Loop through all CV model candidates
    for i, candidate in enumerate(bayes_search_cv.cv_results_["params"]):
        with mlflow.start_run(nested=True, run_name=f"CV_Set_{i}"):
            # Log hyperparameters for this set
            log_params(candidate)

            # Log CV mean accuracy and standard deviation accuracy (variability)
            mean_acc = bayes_search_cv.cv_results_["mean_test_score"][i]
            std_acc = bayes_search_cv.cv_results_["std_test_score"][i]
            log_metric("cv_mean_accuracy", mean_acc)
            log_metric("cv_std_accuracy", std_acc)

            # Optionally, refit and evaluate on test set
            model = bayes_search_cv.estimator.set_params(**candidate)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            log_metric("test_accuracy", accuracy_score(y_test, y_pred))
            log_metric(
                "f1_score", 
                f1_score(y_test, y_pred, average="weighted")
            )

/home/michael/bda602/.venv/lib/python3.14/site-packages/sklearn/ensemble/_iforest.py:343: UserWarning: max_samples (512) is greater than the total number of samples (477). max_samples will be set to n_samples for estimation.
  warn(
/home/michael/bda602/.venv/lib/python3.14/site-packages/sklearn/ensemble/_iforest.py:343: UserWarning: max_samples (512) is greater than the total number of samples (477). max_samples will be set to n_samples for estimation.
  warn(
/home/michael/bda602/.venv/lib/python3.14/site-packages/sklearn/ensemble/_iforest.py:343: UserWarning: max_samples (512) is greater than the total number of samples (478). max_samples will be set to n_samples for estimation.
  warn(
/home/michael/bda602/.venv/lib/python3.14/site-packages/sklearn/ensemble/_iforest.py:343: UserWarning: max_samples (512) is greater than the total number of samples (478). max_samples will be set to n_samples for estimation.
  warn(
/home/michael/bda602/.venv/lib/python3.14/site-packages/sklearn/ense